# Aegis Wave - CSI-Bench Adaptation
## Optimized for CSI-Bench Fall Detection Dataset

### Dataset Characteristics:
- **Source**: CSI-Bench Dataset (FallDetection)
- **Labels**: Fall (0), Nonfall (1)
- **Format**: HDF5 (.h5) files
- **Shape**: (232, 500, 1) [Subcarriers/Antennas, Packets, Channel]
- **Environment**: Real-world human activity sensing

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy import signal
import matplotlib.pyplot as plt
import os
import h5py
from datetime import datetime
from tqdm import tqdm

## 1. Data Loader for CSI-Bench (HDF5)
Loading metadata and reading .h5 files directly.

In [ ]:
def load_csibench_data(metadata_path, base_dir, n_samples=500):
    """
    Load a subset of CSI-Bench data for feasibility testing
    """
    df = pd.read_csv(metadata_path)
    # Sample for quick processing while maintaining class balance
    df_sampled = df.groupby('label').apply(lambda x: x.sample(min(len(x), n_samples//2))).reset_index(drop=True)
    
    X_list = []
    y_list = []
    
    print(f"Loading {len(df_sampled)} samples from {base_dir}...")
    
    for _, row in tqdm(df_sampled.iterrows(), total=len(df_sampled)):
        # The file_path in CSV is relative to the metadata folder or sub_Human
        rel_path = row['file_path']
        if rel_path.startswith('.'):
            rel_path = rel_path[1:]
        if rel_path.startswith('/'):
            rel_path = rel_path[1:]
            
        full_path = os.path.join(base_dir, rel_path)
        
        try:
            with h5py.File(full_path, 'r') as f:
                csi = f['CSI_amps'][:]
                csi = csi.squeeze().T 
                X_list.append(csi)
                y_list.append(0 if row['label'] == 'Fall' else 1)
        except Exception as e:
            continue
            
    return np.array(X_list), np.array(y_list), ['Fall', 'Nonfall']

METADATA_PATH = '../csi-bench-dataset/FallDetection/metadata/sample_metadata.csv'
BASE_DIR = '../csi-bench-dataset/FallDetection/'

X, y, activities = load_csibench_data(METADATA_PATH, BASE_DIR, n_samples=400)
print(f"\nDataset loaded: {X.shape}")
print(f"Activities: {activities}")
print(f"Samples per class: {np.bincount(y)}")

## 2. Preprocessing Pipeline
1. Butterworth filter to remove high-frequency noise
2. Standard normalization

In [ ]:
def butterworth_filter(data, lowcut=0.5, highcut=10, fs=100, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='band')
    
    filtered = np.zeros_like(data)
    for i in range(data.shape[0]):
        for j in range(data.shape[2]):
            filtered[i, :, j] = signal.filtfilt(b, a, data[i, :, j])
    return filtered

print("Applying Butterworth filter...")
X_filtered = butterworth_filter(X)
print("Filtering complete.")

## 3. Train SVM Classifier (Baseline Approach)
Using statistical features (variance, mean, std) as a baseline.

In [ ]:
from sklearn.svm import SVC
from sklearn.decomposition import PCA

def extract_features(data):
    features = []
    for sample in data:
        variance = np.var(sample, axis=0)
        mean = np.mean(sample, axis=0)
        std = np.std(sample, axis=0)
        feature_vec = np.concatenate([variance, mean, std])
        features.append(feature_vec)
    return np.array(features)

X_features = extract_features(X_filtered)
X_train_svm, X_test_svm, y_train_svm, y_test_svm = train_test_split(
    X_features, y, test_size=0.2, random_state=42, stratify=y
)

scaler_svm = StandardScaler()
X_train_scaled = scaler_svm.fit_transform(X_train_svm)
X_test_scaled = scaler_svm.transform(X_test_svm)

pca = PCA(n_components=50)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

svm_model = SVC(kernel='rbf', C=10, probability=True)
svm_model.fit(X_train_pca, y_train_svm)

print(f"SVM Train Accuracy: {svm_model.score(X_train_pca, y_train_svm):.2%}")
print(f"SVM Test Accuracy: {svm_model.score(X_test_pca, y_test_svm):.2%}")

## 4. Train Binary CNN Model
Adapted for (500, 232) input shape.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_filtered, y, test_size=0.2, random_state=42, stratify=y
)

X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)
scaler = StandardScaler()
X_train_norm = scaler.fit_transform(X_train_flat).reshape(X_train.shape)
X_test_norm = scaler.transform(X_test_flat).reshape(X_test.shape)

y_train_cat = keras.utils.to_categorical(y_train, 2)
y_test_cat = keras.utils.to_categorical(y_test, 2)

model = keras.Sequential([
    keras.layers.Conv1D(16, 11, activation='relu', input_shape=(X_train_norm.shape[1], X_train_norm.shape[2])),
    keras.layers.MaxPooling1D(4),
    keras.layers.Conv1D(32, 5, activation='relu'),
    keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.4),
    keras.layers.Dense(2, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
history = model.fit(
    X_train_norm, y_train_cat,
    validation_split=0.2,
    epochs=20,
    batch_size=16,
    verbose=1
)

test_loss, test_acc = model.evaluate(X_test_norm, y_test_cat)
print(f"\nCNN Test Accuracy: {test_acc:.2%}")

## 5. Edge Deployment (TFLite)
Converting to a lightweight format.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('aegis_wave_csibench_edge.tflite', 'wb') as f:
    f.write(tflite_model)

print(f"✅ TFLite model size: {len(tflite_model) / 1024:.2f} KB")

## 6. Visualizing the Difference
Comparing a Fall vs. a Nonfall sample.

In [ ]:
fall_idx = np.where(y_test == 0)[0][0]
nonfall_idx = np.where(y_test == 1)[0][0]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

ax1.plot(X_test_norm[nonfall_idx].mean(axis=1), color='green')
ax1.set_title('Nonfall (Ambient Activity)')
ax1.grid(True, alpha=0.3)

ax2.plot(X_test_norm[fall_idx].mean(axis=1), color='red')
ax2.set_title('FALL DETECTED (High Variance Burst)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()